## 面试问题

同步循环 vs 事件驱动循环：何时让出控制权等外部事件？

## 回答主线

同步循环每步阻塞等待返回，简单但遇长等待（人工审批、异步回调）无法暂停/持久化/迁移；事件驱动循环在需要外部事件处挂起可序列化状态并让出控制权，事件到达后用「状态+事件」恢复。本 Notebook 用需审批的退款循环实现 suspend/resume，验证挂起态可 json 序列化后恢复，并校验事件必须匹配暂停点。

## 真实案例

退款金额 500 超过阈值 200，必须等主管审批。同步循环审批未知时只能阻塞；事件驱动循环挂起 `waiting_for=approval`，审批事件到达后恢复退款。数据为教学事件，不代表真实审批系统。

In [1]:
import json  # 引入 json 序列化挂起态。

refund_request = {"order_id": "od-9", "amount": 500, "threshold": 200}  # 定义金额超阈值需审批的退款请求。
print("退款请求:", refund_request)  # 展示输入请求。
print("是否需要审批:", refund_request["amount"] > refund_request["threshold"])  # 展示金额超阈值需要审批。

退款请求: {'order_id': 'od-9', 'amount': 500, 'threshold': 200}
是否需要审批: True


## 基线（Baseline）

同步循环：审批结果必须一次性预先传入。审批未知时它只能返回阻塞，无法在等待期间挂起并持久化。

In [2]:
def sync_refund(request, approval):  # 同步循环：审批结果必须一次性预先传入。
    if request["amount"] > request["threshold"]:  # 金额超阈值需要审批。
        if approval is None:  # 审批未知时同步循环只能阻塞。
            return "blocked"  # 无法挂起只能返回阻塞。
        if not approval:  # 审批被拒。
            return "rejected"  # 返回拒绝。
    return "refunded"  # 通过或无需审批则退款。

print("同步循环-审批未知:", sync_refund(refund_request, None))  # 展示审批未知时被阻塞。
print("同步循环-审批通过:", sync_refund(refund_request, True))  # 展示必须预先知道审批结果。

同步循环-审批未知: blocked
同步循环-审批通过: refunded


## 核心实现：事件驱动 suspend / resume

推进到审批点时返回可序列化的挂起态并让出控制权；审批事件到达后校验事件与暂停点匹配，再恢复推进。

In [3]:
def run_until_suspend(request):  # 推进到审批点并挂起可序列化状态。
    if request["amount"] > request["threshold"]:  # 金额超阈值触发审批。
        return {"status": "suspended", "waiting_for": "approval", "request": request}  # 挂起并声明等待的事件。
    return {"status": "refunded", "request": request}  # 无需审批直接完成。

def resume(state, event):  # 用外部事件恢复挂起的循环。
    if state.get("waiting_for") != event["type"]:  # 校验事件与暂停点匹配。
        return {"status": "error", "reason": "event_mismatch"}  # 事件错配则拒绝推进。
    if not event["approved"]:  # 审批被拒。
        return {"status": "rejected", "request": state["request"]}  # 返回拒绝。
    return {"status": "refunded", "request": state["request"]}  # 审批通过则完成退款。

In [4]:
suspended = run_until_suspend(refund_request)  # 第一阶段推进到审批点挂起。
print("挂起态:", suspended["status"], "等待事件:", suspended["waiting_for"])  # 展示循环让出控制权。
blob = json.dumps(suspended)  # 把挂起态序列化以便持久化或迁移。
print("挂起态可序列化长度:", len(blob))  # 展示挂起态能被存储。
loaded = json.loads(blob)  # 从存储恢复挂起态。
approval_event = {"type": "approval", "approved": True}  # 审批事件到达。
final = resume(loaded, approval_event)  # 用事件恢复并推进到完成。
print("恢复后状态:", final["status"])  # 展示事件驱动恢复到退款完成。

挂起态: suspended 等待事件: approval
挂起态可序列化长度: 116
恢复后状态: refunded


## 结果解读

循环在审批点挂起为 `suspended`，挂起态被 json 序列化（可存库/迁移），审批事件到达后 resume 推进到 `refunded`。相比同步循环「审批未知即 blocked」，事件驱动把等待期变成可持久化的挂起态。

## 失败案例与修正

恢复不能盲目继续：迟到或错配的事件必须被拒绝。下面用不匹配暂停点的 `cancel` 事件恢复得到 error，用审批被拒事件恢复走 rejected 路径——证明 resume 会校验事件匹配暂停点。

In [5]:
mismatch_event = {"type": "cancel", "approved": True}  # 构造与暂停点不匹配的事件。
bad = resume(loaded, mismatch_event)  # 用错配事件尝试恢复。
print("错配事件恢复:", bad["status"], bad.get("reason"))  # 展示恢复会校验事件匹配暂停点。
reject_event = {"type": "approval", "approved": False}  # 构造审批被拒事件。
rejected = resume(loaded, reject_event)  # 用拒绝事件恢复。
print("审批被拒恢复:", rejected["status"])  # 展示拒绝路径也能正确处理。

错配事件恢复: error event_mismatch
审批被拒恢复: rejected


In [6]:
assert sync_refund(refund_request, None) == "blocked"  # 同步循环在审批未知时被阻塞。
assert suspended["status"] == "suspended"  # 事件驱动循环应挂起等待。
assert suspended["waiting_for"] == "approval"  # 挂起态应声明等待的事件类型。
assert json.loads(blob) == suspended  # 挂起态应可序列化 round-trip。
assert final["status"] == "refunded"  # 审批事件到达后应恢复到完成。
assert bad["status"] == "error"  # 错配事件应被拒绝。
assert rejected["status"] == "rejected"  # 审批被拒应走拒绝路径。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
